# Skymapper import



In [ ]:

import pandas as pd

from hats_import.pipeline import pipeline_with_client, pipeline
from hats_import.catalog.arguments import ImportArguments
from hats_import.catalog.file_readers import CsvReader, InputReader
from hats_import import CollectionArguments, VerificationArguments
import pyarrow as pa
import pyarrow.parquet as pq
import os
import hats_import
from dask.distributed import Client
from pathlib import Path

In [ ]:

hats_import.__version__

In [ ]:
# !mkdir /epyc/data3/hats/catalogs/v09/skymapper

In [ ]:
schema_file = os.path.join("/epyc/data3/hats/raw/skymapper/", "schema_from_pyarrow.parquet")

In [ ]:
with Client(
        local_directory="/epyc/data3/hats/tmp/",
        n_workers=30,
        threads_per_worker=1,
    ) as client:
    args = ImportArguments(
        output_artifact_name="sky_mapper_dr4",
        input_path="/epyc/data3/hats/raw/skymapper/dr4_master",
        output_path="/epyc/data3/hats/catalogs/v09/skymapper",

        use_schema_file=schema_file,
        file_reader=CsvReader(
            chunksize=250_000,
            compression="gzip",
            schema_file=schema_file,
        ),
        ra_column='raj2000',
        dec_column='dej2000',
        
        expected_total_rows=724_329_344,
        pixel_threshold= 5_000_000,
        highest_healpix_order=8,
        drop_empty_siblings=True,

        skymap_alt_orders=[2, 4, 6],
        row_group_kwargs={"num_rows": 500_000},
    
        completion_email_address="delucchi@andrew.cmu.edu",
        progress_bar=True,
        simple_progress_bar=True,        
    )
    pipeline_with_client(args, client)

In [ ]:
with Client(
        local_directory="/epyc/data3/hats/tmp/",
        n_workers=30,
        threads_per_worker=1,
    ) as client:
    args = ImportArguments(
        output_artifact_name="sky_mapper_dr4",
        input_path="/epyc/data3/hats/raw/skymapper/dr4_master",
        output_path="/epyc/data3/hats/catalogs/v09/skymapper",

        use_schema_file=schema_file,
        file_reader=CsvReader(
            chunksize=250_000,
            compression="gzip",
            schema_file=schema_file,
        ),
        ra_column='raj2000',
        dec_column='dej2000',
        
        expected_total_rows=724_329_344,
        pixel_threshold= 5_000_000,
        highest_healpix_order=8,
        drop_empty_siblings=True,

        skymap_alt_orders=[2, 4, 6],
        row_group_kwargs={"num_rows": 500_000},
    
        completion_email_address="delucchi@andrew.cmu.edu",
        progress_bar=True,
        simple_progress_bar=True,        
    )
    pipeline_with_client(args, client)

In [ ]:
args = (
    CollectionArguments(
        completion_email_address="delucchi@andrew.cmu.edu",
        output_artifact_name="skymapper",
        output_path="/epyc/data3/hats/catalogs/v09",
        progress_bar=True,
        simple_progress_bar=True,
    )
    .catalog(        output_artifact_name="sky_mapper_dr4"    )
    .add_margin(margin_threshold=5.0, is_default=True)
)

In [ ]:
with Client(
        local_directory="/epyc/data3/hats/tmp/",
        n_workers=30,
        threads_per_worker=1,
    ) as client:
    pipeline_with_client(args, client)

In [ ]:
args = VerificationArguments(
    input_catalog_path="/epyc/data3/hats/catalogs/v09/skymapper",
    output_path="./verification/skymapper_dr4",
)
pipeline(args)

## Photometry

(sources, detections, whatever you want to call it).

In [ ]:
schema_file = os.path.join("/epyc/data3/hats/raw/skymapper/", "photo_schema_from_pyarrow.parquet")

In [ ]:
schema_from_pyarrow = pa.schema(
    [
pa.field('object_id', pa.int64()),
pa.field('image_id', pa.int64()),
pa.field('ccd', pa.int32()),
pa.field('ra_img', pa.float64()),
pa.field('decl_img', pa.float64()),
pa.field('x_img', pa.float64(), nullable=True),
pa.field('y_img', pa.float64(), nullable=True),
pa.field('flags', pa.int32(), nullable=True),
pa.field('nimaflags', pa.int64(), nullable=True),
pa.field('background', pa.float64(), nullable=True),
pa.field('flux_max', pa.float64(), nullable=True),
pa.field('mu_max', pa.float64(), nullable=True),
pa.field('class_star', pa.float64(), nullable=True),
pa.field('a', pa.float64(), nullable=True),
pa.field('e_a', pa.float64(), nullable=True),
pa.field('b', pa.float64(), nullable=True),
pa.field('e_b', pa.float64(), nullable=True),
pa.field('pa', pa.float64(), nullable=True),
pa.field('e_pa', pa.float64(), nullable=True),
pa.field('elong', pa.float64(), nullable=True),
pa.field('fwhm', pa.float64(), nullable=True),
pa.field('radius_petro', pa.float64(), nullable=True),
pa.field('radius_kron', pa.float64(), nullable=True),
pa.field('radius_frac20', pa.float64(), nullable=True),
pa.field('radius_frac50', pa.float64(), nullable=True),
pa.field('radius_frac90', pa.float64(), nullable=True),
pa.field('flux_petro', pa.float64(), nullable=True),
pa.field('e_flux_petro', pa.float64(), nullable=True),
pa.field('mag_petro', pa.float64(), nullable=True),
pa.field('e_mag_petro', pa.float64(), nullable=True),
pa.field('flux_kron', pa.float64(), nullable=True),
pa.field('e_flux_kron', pa.float64(), nullable=True),
pa.field('mag_kron', pa.float64(), nullable=True),
pa.field('e_mag_kron', pa.float64(), nullable=True),
pa.field('flux_ap02', pa.float64(), nullable=True),
pa.field('e_flux_ap02', pa.float64(), nullable=True),
pa.field('mag_apc02', pa.float64(), nullable=True),
pa.field('e_mag_apc02', pa.float64(), nullable=True),
pa.field('flux_ap03', pa.float64(), nullable=True),
pa.field('e_flux_ap03', pa.float64(), nullable=True),
pa.field('mag_apc03', pa.float64(), nullable=True),
pa.field('e_mag_apc03', pa.float64(), nullable=True),
pa.field('flux_ap04', pa.float64(), nullable=True),
pa.field('e_flux_ap04', pa.float64(), nullable=True),
pa.field('mag_apc04', pa.float64(), nullable=True),
pa.field('e_mag_apc04', pa.float64(), nullable=True),
pa.field('flux_ap05', pa.float64(), nullable=True),
pa.field('e_flux_ap05', pa.float64(), nullable=True),
pa.field('mag_apc05', pa.float64(), nullable=True),
pa.field('e_mag_apc05', pa.float64(), nullable=True),
pa.field('flux_ap06', pa.float64(), nullable=True),
pa.field('e_flux_ap06', pa.float64(), nullable=True),
pa.field('mag_apc06', pa.float64(), nullable=True),
pa.field('e_mag_apc06', pa.float64(), nullable=True),
pa.field('flux_ap08', pa.float64(), nullable=True),
pa.field('e_flux_ap08', pa.float64(), nullable=True),
pa.field('mag_apc08', pa.float64(), nullable=True),
pa.field('e_mag_apc08', pa.float64(), nullable=True),
pa.field('flux_ap10', pa.float64(), nullable=True),
pa.field('e_flux_ap10', pa.float64(), nullable=True),
pa.field('mag_apc10', pa.float64(), nullable=True),
pa.field('e_mag_apc10', pa.float64(), nullable=True),
pa.field('flux_ap15', pa.float64(), nullable=True),
pa.field('e_flux_ap15', pa.float64(), nullable=True),
pa.field('mag_apr15', pa.float64(), nullable=True),
pa.field('e_mag_apr15', pa.float64(), nullable=True),
pa.field('flux_ap20', pa.float64(), nullable=True),
pa.field('e_flux_ap20', pa.float64(), nullable=True),
pa.field('mag_apr20', pa.float64(), nullable=True),
pa.field('e_mag_apr20', pa.float64(), nullable=True),
pa.field('flux_ap30', pa.float64(), nullable=True),
pa.field('e_flux_ap30', pa.float64(), nullable=True),
pa.field('mag_apr30', pa.float64(), nullable=True),
pa.field('e_mag_apr30', pa.float64(), nullable=True),
pa.field('imaflags', pa.int64(), nullable=True),
pa.field('x_mosaic', pa.float64(), nullable=True),
pa.field('y_mosaic', pa.float64(), nullable=True),
pa.field('chi2_psf', pa.float64(), nullable=True),
pa.field('flux_psf', pa.float64(), nullable=True),
pa.field('e_flux_psf', pa.float64(), nullable=True),
pa.field('mag_psf', pa.float64(), nullable=True),
pa.field('e_mag_psf', pa.float64(), nullable=True),
pa.field('filter', pa.string()),
pa.field('img_qual', pa.int32(), nullable=True),
pa.field('use_in_clipped', pa.int32(), nullable=True),
pa.field('object_id_local', pa.int64(), nullable=True),
    ]
)

In [ ]:
class SkymapperPhotometryCsvReader(InputReader):
    
    def __init__(self, column_names=None, convert_options=None,read_options=None, **kwargs):
        self.kwargs = kwargs
        self.column_names = column_names
        self.convert_options = convert_options or csv.ConvertOptions()
        self.read_options = read_options or csv.ReadOptions(block_size=1048576*10)

        
    def read(self, input_file, read_columns=None):
        self.regular_file_exists(input_file, **self.kwargs)

        read_columns = read_columns or self.column_names
        convert_options = self.convert_options
        if read_columns:
            convert_options.include_columns = read_columns
        with csv.open_csv(input_file, convert_options=convert_options, read_options=self.read_options, **self.kwargs) as reader:
            for next_chunk in reader:
                if next_chunk is None:
                    break
                table = pa.Table.from_batches([next_chunk])
                table = table.replace_schema_metadata()
                yield table

In [ ]:
with Client(
        local_directory="/epyc/data3/hats/tmp/",
        n_workers=30,
        threads_per_worker=1,
    ) as client:
    args = ImportArguments(
        output_artifact_name="photometry",
        input_file_list=list(Path("/epyc/data3/hats/raw/skymapper/photometry").glob("SkyMapper.DR4.pho*.csv.gz")),
        output_path="/epyc/data3/hats/catalogs/v09/skymapper",

        use_schema_file=schema_file,
        file_reader=SkymapperPhotometryCsvReader(convert_options = csv.ConvertOptions(column_types = schema_from_pyarrow)),
        ra_column="ra_img",
        dec_column="decl_img",
        
        expected_total_rows=15_101_099_413,
        pixel_threshold= 3_000_000,
        highest_healpix_order=10,
        drop_empty_siblings=True,

        skymap_alt_orders=[2, 4, 6, 8],
        row_group_kwargs={"num_rows": 500_000},
    
        completion_email_address="delucchi@andrew.cmu.edu",
        progress_bar=True,
        simple_progress_bar=True,        
    )
    pipeline_with_client(args, client)

In [ ]:
args = VerificationArguments(
    input_catalog_path="/epyc/data3/hats/catalogs/v09/skymapper/photometry",
    output_path="./verification/skymapper_photometry",
)
pipeline(args)